In [1]:
%pip install pandas
%pip install numpy
%pip install matplotlib
%pip install seaborn
%pip install plotly

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [49]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

f = Path.cwd().__str__() + "\\data_sources\\marketing\\ifood_df.csv"

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

market_df = pd.read_csv(f)

print(market_df.columns)
print(market_df.head())


Index(['Income', 'Kidhome', 'Teenhome', 'Recency', 'MntWines', 'MntFruits',
       'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts',
       'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases',
       'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth',
       'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1',
       'AcceptedCmp2', 'Complain', 'Z_CostContact', 'Z_Revenue', 'Response',
       'Age', 'Customer_Days', 'marital_Divorced', 'marital_Married',
       'marital_Single', 'marital_Together', 'marital_Widow',
       'education_2n Cycle', 'education_Basic', 'education_Graduation',
       'education_Master', 'education_PhD', 'MntTotal', 'MntRegularProds',
       'AcceptedCmpOverall'],
      dtype='str')
    Income  Kidhome  Teenhome  Recency  MntWines  MntFruits  MntMeatProducts  \
0  58138.0        0         0       58       635         88              546   
1  46344.0        1         1       38        11          1                6   
2  71613

In [50]:
avg_age = round(market_df['Age'].mean(), 2)
print(f"Average age in marketing data: {avg_age}")

#Create age groups column
age_bins = [0, 12, 19, 29, 60, 100]
age_labels = ['0-12 (Child)', '13-19 (Teenager)', '20-30 (Young Adult)', '31-60 (Adult)', '61-100+ (Senior)']

market_df['Age Group'] = pd.cut(market_df['Age'], bins=age_bins, labels=age_labels)

#Average income by age group
age_income_avg = market_df.groupby('Age Group')[['Income']].mean().rename(columns={'Income': 'Average Income'})
age_income_avg['Average Income'] = age_income_avg['Average Income'].round(2)

#Print the following variable to confirm Age Group is bucketed correctly
age_check = market_df.join(age_income_avg, how='inner', on='Age Group')[['Age', 'Age Group']].sort_values(by='Age')

avg_income = round(market_df['Income'].mean(), 2)

print(f"Average age in marketing data: {avg_age}\nAvergage income in marketing data: {avg_income}")


Average age in marketing data: 51.1
Average age in marketing data: 51.1
Avergage income in marketing data: 51622.09


In [51]:
conditions = [(market_df['education_2n Cycle'] == 1), (market_df['education_Basic'] == 1), (market_df['education_Graduation'] == 1), (market_df['education_Master'] == 1), (market_df['education_PhD'] == 1)]

choices = ['Masters', 'K-12', 'Bachelors', 'Masters', 'PhD']
default_value = 'Other'

market_df['Education Category'] = np.select(conditions, choices, default=default_value)

conditions = [(market_df['marital_Single'] == 1), (market_df['marital_Married'] == 1),(market_df['marital_Divorced'] == 1), (market_df['marital_Widow'] == 1), (market_df['marital_Together'] == 1)]

choices = ['Single', 'Married', 'Divorced', 'Widow', 'In a relationship']
default_value = 'Other'

market_df['Marital Category'] = np.select(conditions, choices, default=default_value)

columns_to_drop = ['education_2n Cycle', 'education_Basic', 'education_Graduation', 'education_Master', 'education_PhD', 'marital_Single', 'marital_Divorced', 'marital_Widow', 'marital_Together', 'marital_Married']
market_df.drop(columns=columns_to_drop, inplace=True)

print(market_df.head())

    Income  Kidhome  Teenhome  Recency  MntWines  MntFruits  MntMeatProducts  \
0  58138.0        0         0       58       635         88              546   
1  46344.0        1         1       38        11          1                6   
2  71613.0        0         0       26       426         49              127   
3  26646.0        1         0       26        11          4               20   
4  58293.0        1         0       94       173         43              118   

   MntFishProducts  MntSweetProducts  MntGoldProds  NumDealsPurchases  \
0              172                88            88                  3   
1                2                 1             6                  2   
2              111                21            42                  1   
3               10                 3             5                  2   
4               46                27            15                  5   

   NumWebPurchases  NumCatalogPurchases  NumStorePurchases  NumWebVisitsMonth  \

In [56]:
###Chart and Graphs for the Marketing Data###
#This bar chart is for Marketing data is to display the Total Campaign Responses by Age Group
age_group_responses_summary = market_df.groupby('Age Group').agg({'Response': 'sum'}).reset_index()

fig = px.bar(age_group_responses_summary, x='Age Group', y='Response', title='Age Group by Total Campaign Responses', text_auto=True, color_discrete_sequence=['white'], labels={'Age Group': 'Age Group', 'Response': 'Total Campaign Responses'})
fig.show()


In [57]:
#This bar chart displays the relationship between Education Category and Total Campaign Responses
#With Average age as the trend line

education_category_responses_summary = market_df.groupby('Education Category').agg({'Response': 'sum', 'Age': 'mean'}).reset_index()

custom_order = ['K-12', 'Bachelors', 'Masters', 'PhD']

order_map = {item: i for i, item in enumerate(custom_order)}

education_category_responses_summary = education_category_responses_summary.sort_values(by='Education Category', key=lambda x: x.map(order_map))

bar_chart = go.Bar(
    x=education_category_responses_summary['Education Category'],
    y=education_category_responses_summary['Response'],
    name='Total Campaign Responses',
    marker_color='white'
)


trendline = go.Scatter(
    x=education_category_responses_summary['Education Category'],
    y=education_category_responses_summary['Age'],
    mode='lines+markers',
    name='Average Age',
    line=dict(color='blue', width=2)
)


fig = go.Figure(data=[bar_chart, trendline])


fig.update_layout(

    title='Education Category by Total Campaign Responses with Average Age Trend Line',
    xaxis_title='Education Category',
    yaxis_title='Total Campaign Responses',
)

fig.show()


  Education Category  Response        Age
0          Bachelors       152  50.381851
1               K-12         2  42.537037
2            Masters        78  51.000000
3                PhD       101  53.848739


In [70]:
#This is the scatter plot of the correlation between the Total of number of Catalog Purchases versus the Total number of purchases made in store by Age

fig = px.scatter(market_df, y="NumStorePurchases", x="NumCatalogPurchases", color="Age",
                 title="Total number of Catalog Purchases versus Total number of in-store Purchases by Age", hover_data=['Age'], labels={
        'NumStorePurchases': 'Total number of in-store Purchases', 'NumCatalogPurchases': 'Total number of Catalog Purchases'})

fig.show()